In [1]:
!pip install -q transformers datasets peft torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 74.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 63.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 28.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 11.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 68.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 11.7 MB/s eta 0:00:00
ERROR: pip's dependency 

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import get_peft_model, LoraConfig, TaskType
from datasets import Dataset  
from transformers import TrainingArguments, Trainer
import gc

2025-08-04 15:00:45.882343: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754319646.114463      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754319646.180675      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [4]:
summarization_data = [
    {
        "text": "Universitas Indonesia (UI) has its main campus in Depok, West Java. It also maintains a secondary campus in Salemba, Central Jakarta, which is primarily dedicated to the Faculty of Medicine and the Faculty of Dentistry.",
        "summary": "Universitas Indonesia (UI) is primarily located in Depok, with a secondary campus in Salemba, Jakarta for medical studies."
    },
    {
        "text": "Institut Teknologi Bandung (ITB) features its main and most historic campus on Jalan Ganesha, Bandung, West Java. To accommodate growth, it has expanded with a secondary campus in Jatinangor, Sumedang Regency, which houses several of a newer faculties and programs.",
        "summary": "Institut Teknologi Bandung (ITB) has its main campus in Bandung and a newer, secondary campus for various faculties in Jatinangor, Sumedang."
    },
    {
        "text": "Universitas Gadjah Mada (UGM) is located in the Bulaksumur area within the Sleman Regency, Special Region of Yogyakarta. Its sprawling campus is considered a landmark and is centrally located within the vibrant student city of Yogyakarta.",
        "summary": "Universitas Gadjah Mada (UGM) is a major university centrally located in the city of Yogyakarta, specifically in the Bulaksumur area, Sleman."
    },
    {
        "text": "Institut Pertanian Bogor (IPB University) has its primary and largest campus located in Dramaga, Bogor Regency, West Java, which is a comprehensive academic and research hub. A smaller, more accessible campus is located in Baranangsiang, Bogor City, which is home to its prestigious business school.",
        "summary": "IPB University's main campus is in Dramaga, Bogor, while its business school is located at a secondary campus in the Baranangsiang area of Bogor City."
    },
    {
        "text": "Universitas Airlangga (Unair) is a prominent university located in Surabaya, East Java. Its facilities are spread across three main campus locations within the city: Campus A for Medicine, Campus B for Social Sciences, and Campus C for Science and Technology.",
        "summary": "Universitas Airlangga (Unair) is located in Surabaya and operates across three main campuses (A, B, and C), each focusing on different fields of study."
    },
    {
        "text": "Institut Teknologi Sepuluh Nopember (ITS) has its main campus situated in the Sukolilo district of Surabaya, East Java. Additionally, it operates a secondary campus in the Manyar area of Surabaya, which is specifically dedicated to the Faculty of Civil, Environmental, and Geo Engineering.",
        "summary": "The main campus of Institut Teknologi Sepuluh Nopember (ITS) is in Sukolilo, Surabaya, with a secondary campus in Manyar for Civil Engineering."
    }
]

In [5]:
# Load model dan tokenizer

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading model... (this may take a while)")
# Load pre-trained model & tokenizer for fine-tuning
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Move to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("Model loaded successfully!")


Loading model... (this may take a while)


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Model loaded successfully!


In [6]:
# Data preprocessing function
def format_summarization_prompt(example):
    return {
        "text": f"Summarize the following text:\n\n{example['text']}\n\nSummary: {example['summary']}"
    }

summary_dataset = Dataset.from_list(summarization_data)
formatted_dataset = summary_dataset.map(format_summarization_prompt)

# Tokenization function
def preprocess_function(examples):
    inputs = tokenizer(
        examples['text'],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    inputs["labels"] = inputs["input_ids"].copy()
    return inputs

tokenized_dataset = formatted_dataset.map(preprocess_function, batched=True)

# Define LoRA configuration
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

# Wrap model with LoRA
model = get_peft_model(model, lora_config)

# Set Training Arguments
training_args = TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=20,
    max_steps=150,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    output_dir="outputs_summarization_bot",
    report_to="none",
    remove_unused_columns=False,
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

# Clear cache and start training
gc.collect()
torch.cuda.empty_cache()
print("--- Starting Fine-Tuning for Summarization Task ---")
trainer.train()

# Save the fine-tuned model
fine_tuned_model_path = "fine-tuned-Summarization-Bot"
model.save_pretrained(fine_tuned_model_path)
tokenizer.save_pretrained(fine_tuned_model_path)
print(f"--- Fine-tuned summarization model saved to {fine_tuned_model_path} ---")


Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


--- Starting Fine-Tuning for Summarization Task ---


Step,Training Loss
10,14.299100
20,5.338800
30,0.469800
40,0.327100
50,0.239600
60,0.163600
70,0.092800
80,0.038800
90,0.018100
100,0.011500


--- Fine-tuned summarization model saved to fine-tuned-Summarization-Bot ---


In [7]:
#!zip -r fine-tuned-Summarization-Bot.zip fine-tuned-Summarization-Bot

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  adding: fine-tuned-Summarization-Bot/ (stored 0%)
  adding: fine-tuned-Summarization-Bot/adapter_config.json (deflated 54%)
  adding: fine-tuned-Summarization-Bot/tokenizer.json (deflated 85%)
  adding: fine-tuned-Summarization-Bot/adapter_model.safetensors (deflated 8%)
  adding: fine-tuned-Summarization-Bot/tokenizer_config.json (deflated 69%)
  adding: fine-tuned-Summarization-Bot/README.md (deflated 66%)
  adding: fine-tuned-Summarization-Bot/special_tokens_map.json (deflated 79%)
  adding: fine-tuned-Summarization-Bot/tokenizer.model (deflated 55%)
  adding: fine-tuned-Summarization-Bot/chat_template.jinja (deflated 60%)


In [8]:
fine_tuned_model_path = "fine-tuned-Summarization-Bot" 

print(f"\n--- 3. Testing the Fine-Tuned Summarization Model from {fine_tuned_model_path} ---")

ft_model = AutoModelForCausalLM.from_pretrained(fine_tuned_model_path)
ft_tokenizer = AutoTokenizer.from_pretrained(fine_tuned_model_path)

device = "cuda" if torch.cuda.is_available() else "cpu"
ft_model.to(device)

def generate_summary(text_to_summarize, max_length=100):
    """
    Fungsi ini mengambil teks panjang, memformatnya menjadi prompt,
    dan menghasilkan ringkasan dari model.
    """
    # Prompt hanya berisi instruksi dan teks, tanpa jawaban
    prompt = f"Summarize the following text:\n\n{text_to_summarize}\n\nSummary:"
    
    inputs = ft_tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = ft_model.generate(
            **inputs, 
            max_new_tokens=max_length,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            eos_token_id=ft_tokenizer.eos_token_id
        )
    
    response = ft_tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    try:
        summary = response.split("Summary:")[1].strip()
    except IndexError:
        summary = "Model failed to generate a valid summary."
        
    return summary

# Example Test Cases - Known Questions (from training data)
print("\n--- Testing Known Text (from training data) ---")
test_texts_known = [
    "Institut Teknologi Bandung (ITB) features its main and most historic campus on Jalan Ganesha, Bandung, West Java. To accommodate growth, it has expanded with a secondary campus in Jatinangor, Sumedang Regency, which houses several of a newer faculties and programs.",
    "Universitas Airlangga (Unair) is a prominent university located in Surabaya, East Java. Its facilities are spread across three main campus locations within the city: Campus A for Medicine, Campus B for Social Sciences, and Campus C for Science and Technology."
]

for i, text in enumerate(test_texts_known):
    print(f"\n[Known Text #{i+1}]")
    print(f"Original Text:\n{text}")
    summary = generate_summary(text, max_length=60)
    print(f"\n--> Generated Summary:\n{summary}\n")


# Example Test Cases - Unknown Questions 
print("\n--- Testing Unknown Text (to check generalization) ---")
test_text_unknown = "Universitas Brawijaya (UB) is a major public university located in the city of Malang, East Java. Known for its green and spacious campus environment, it offers a wide range of programs across many faculties, from medicine and engineering to social sciences and agriculture, making it a popular choice for students across Indonesia."

print(f"\n[Unknown Text - Universitas Brawijaya]")
print(f"Original Text:\n{test_text_unknown}")
summary = generate_summary(test_text_unknown, max_length=50)
print(f"\n--> Generated Summary:\n{summary}\n")


# Example Test Cases - Wrong/External Question
print("\n--- Testing Wrong/External Text (to check robustness) ---")
# Teks baru dengan topik yang sama sekali berbeda
test_text_external = "The Komodo dragon is a species of lizard found in the Indonesian islands of Komodo, Rinca, Flores, and Gili Motang. A member of the monitor lizard family, it is the largest living species of lizard, growing to a maximum length of 3 metres (10 ft) in rare cases and weighing up to approximately 70 kilograms (150 lb)."

print(f"\n[External Text - Komodo Dragon]")
print(f"Original Text:\n{test_text_external}")
summary = generate_summary(test_text_external, max_length=50)
print(f"\n--> Generated Summary:\n{summary}\n")



--- 3. Testing the Fine-Tuned Summarization Model from fine-tuned-Summarization-Bot ---

--- Testing Known Text (from training data) ---

[Known Text #1]
Original Text:
Institut Teknologi Bandung (ITB) features its main and most historic campus on Jalan Ganesha, Bandung, West Java. To accommodate growth, it has expanded with a secondary campus in Jatinangor, Sumedang Regency, which houses several of a newer faculties and programs.

--> Generated Summary:
Institut Teknologi Bandung (ITB) has its main campus in Bandung and a newer, secondary campus for various faculties in Jatinangor, Sumedang.


[Known Text #2]
Original Text:
Universitas Airlangga (Unair) is a prominent university located in Surabaya, East Java. Its facilities are spread across three main campus locations within the city: Campus A for Medicine, Campus B for Social Sciences, and Campus C for Science and Technology.

--> Generated Summary:
Universitas Airlangga (Unair) is located in Surabaya and operates across three mai